# RAG Database Explorer

Connects to the local `rag_db` PostgreSQL database and explores ingested documents.

In [14]:
import os
import pandas as pd
import psycopg
from dotenv import load_dotenv

load_dotenv()

conn = psycopg.connect(
    f"host={os.getenv('DB_HOST')} dbname={os.getenv('DB_NAME')} "
    f"user={os.getenv('DB_USER')} password={os.getenv('DB_PASSWORD')}"
)
print("Connected.")

Connected.


## Structure of the `documents` table

In [23]:
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'documents'
    ORDER BY ordinal_position
""", conn)

/tmp/ipykernel_21043/2074958869.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,column_name,data_type
0,id,bigint
1,content,text
2,metadata,jsonb
3,embedding,USER-DEFINED


## Load all documents

In [15]:
df = pd.read_sql(
    "SELECT id, content, metadata FROM documents ORDER BY id",
    conn,
)

# Flatten metadata JSONB into columns
meta = pd.json_normalize(df["metadata"])
df = pd.concat([df.drop(columns=["metadata"]), meta], axis=1)

df["content_length"] = df["content"].str.len()
df["word_count"] = df["content"].str.split().str.len()

print(f"{len(df)} documents loaded.")
df.head()

/tmp/ipykernel_21043/2978462528.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


2038 documents loaded.


,id,content,path,source,source_id,original_name,content_length,word_count
0,36,http://www.reuters.com/article/us-europe-robot...,,dropbox,,Robots.docx,597,14
1,37,A changer :\n9. refaire calcul de jours de pré...,,dropbox,,Questions citoyenneté.docx,210,30
2,38,Used to test globbing.,,dropbox,,test_file.txt,22,4
3,39,Les principes macroéconomique de base des écon...,/others/uqam/cours/autres cours/macro - eco 24...,dropbox,id:HxvW0XNIQQAAAAAAAAAAAQ,6.1 Economie ouverte I_version longue.pptx,13144,2173
4,40,"Pierre-Yves Yanni\n4577 Rue Cartier, Montréal,...",/job market/old job markets/job market 2015-18...,dropbox,id:PNvfl_uYWcQAAAAAAAMSow,Pierre_Yanni_CV_BdC_Nov_2016.docx,1686,201


## Overview

In [16]:
print("=== Summary ===")
print(f"Total documents : {len(df)}")
print(f"Total words     : {df['word_count'].sum():,}")
print(f"Avg words/doc   : {df['word_count'].mean():.0f}")
print()
print("=== By source ===")
print(df["source"].value_counts().to_string())
print()
print("=== Word count distribution ===")
print(df["word_count"].describe().to_string())

=== Summary ===
Total documents : 2038
Total words     : 5,894,395
Avg words/doc   : 2892

=== By source ===
source
dropbox    1546
drive       492

=== Word count distribution ===
count      2038.000000
mean       2892.244848
std       28822.326740
min           1.000000
25%          58.250000
50%         192.000000
75%         588.000000
max      693417.000000


## Document list

In [17]:
df[["id", "source", "original_name", "word_count"]].sort_values("word_count", ascending=False)

,id,source,original_name,word_count
596,632,dropbox,SHIPHIST_label.csv.txt,693417
357,393,dropbox,SHIPHIST_label.csv.txt,693417
1327,1363,dropbox,crsto10.txt,466300
1241,1277,dropbox,crsto10.txt,466300
1375,1411,dropbox,2701-0.txt,215830
...,...,...,...,...
944,980,dropbox,top_level.txt,1
1117,1153,dropbox,test_ppl.txt,1
1121,1157,dropbox,p022_names.txt,1
1098,1134,dropbox,test_ppl.txt,1


## Preview document content

In [18]:
# Change doc_id to inspect a specific document
doc_id = df["id"].iloc[0]

row = df[df["id"] == doc_id].iloc[0]
print(f"Name   : {row['original_name']}")
print(f"Source : {row['source']}")
print(f"Words  : {row['word_count']}")
print()
print(row["content"][:2000])

Name   : Robots.docx
Source : dropbox
Words  : 14

http://www.reuters.com/article/us-europe-robots-lawmaking/european-parliament-calls-for-robot-law-rejects-robot-tax-idUSKBN15V2KM
Should we tax robots: http://www.kellogg.northwestern.edu/faculty/rebelo/htm/robots.pdf
Tax policy: https://poseidon01.ssrn.com/delivery.php?ID=971089017066124090103096106066086088004051077072079059077026090111001016027120023127011038049127007036048084116088008023010108024011023022080066079000000013093097088066029015024105116105096127107121075083004122124013097112064098098107022126119118005117&EXT=pdf
Robots and jobs: evidence http://economics.mit.edu/files/12763


## Semantic search

In [19]:
from pgvector.psycopg import register_vector
from sentence_transformers import SentenceTransformer

register_vector(conn)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded.")

/home/pyanni/projects/my-llm-writing-project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model loaded.


In [21]:
query = "problem with girlfriend"
top_k = 5

embedding = model.encode(query).tolist()

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT id, metadata->>'original_name' AS name, metadata->>'source' AS source,
               LEFT(content, 300) AS preview,
               embedding <-> %s::vector AS distance
        FROM documents
        ORDER BY distance
        LIMIT %s
        """,
        (embedding, top_k),
    )
    results = cur.fetchall()

results_df = pd.DataFrame(results, columns=["id", "name", "source", "preview", "distance"])
results_df

,id,name,source,preview,distance
0,1640,Should I stay or should I go?,drive,Pro\nStill love her\nCon\nShe would prefer to ...,1.149932
1,2053,Should I stay or should I go.docx,drive,Pro\nStill love her\nCon\nShe would prefer to ...,1.149932
2,1141,Marina.docx,dropbox,"Dear Marina,\nYou showed me a side of you that...",1.188233
3,1353,Marina.docx,dropbox,"Dear Marina,\nYou showed me a side of you that...",1.188233
4,1660,McKinsey Hanh.docx,drive,To do\nsend her CV and cover letter (ask again...,1.268157


## Close connection

In [7]:
conn.close()
print("Connection closed.")

Connection closed.
